# 02. External Web Data Extraction

This notebook focuses on parsing the historical web crawling datasets to extract and standardize external metrics like ratings, reviews, and social engagement.

## Key Fixes Implemented Here:
- **Google Maps Direct Integration**: Replaced the Apify dump with the user-provided `gmaps_meta.csv`, directly linking ratings and reviews to our verified `restaurant_id`.
- **Social Media Fuzzy Mapping**: Utilizes fuzzy matching to link scraped TikTok metrics and Facebook/Wongnai ratings from Google search JSON dumps to our internal `restaurant_name_en`.

In [1]:
import pandas as pd
import numpy as np
import os
import ast
import re
from rapidfuzz import process
import warnings
warnings.filterwarnings('ignore')

## 1. Load Master Cleaned Data

In [2]:
OUTPUT_DIR = "Output_Data"
MASTER_FILE = os.path.join(OUTPUT_DIR, "master_cleaned_data.csv")

print("Loading Master Data...")
df_master = pd.read_csv(MASTER_FILE)
print(f"Loaded {len(df_master)} master records.")

# Create fuzzy mapping choices
choices = df_master['restaurant_name_en'].dropna().unique()

Loading Master Data...
Loaded 3727 master records.


## 2. Google Maps Direct Merge

Reads `gmaps_meta.csv` directly into the dataframe, bypassing the Apify fuzzy matching altogether for 100% accuracy.

In [3]:
print("\n--- Processing Google Maps ---")
df_gmaps = pd.read_csv("../gmaps_meta.csv")

# Rename to our standard columns
df_gmaps.rename(columns={'scraped_rating': 'gmaps_rating', 'scraped_review_count': 'gmaps_reviews'}, inplace=True)

# Deduplicate just in case
df_gmaps = df_gmaps[['restaurant_id', 'gmaps_rating', 'gmaps_reviews']].drop_duplicates(subset=['restaurant_id'])
print(f"Extracted {len(df_gmaps)} unique Google Maps records.")


--- Processing Google Maps ---
Extracted 375 unique Google Maps records.


## 3. TikTok Processing

Aggregates hashtags and fuzzy matches them back to our master names.

In [4]:
print("\n--- Processing TikTok ---")
df_tk = pd.read_csv("Apify/apify_tiktok_test.csv")

def extract_hashtag(url):
    try:
        if 'tag/' in str(url):
            return str(url).split('tag/')[1].split('?')[0].lower()
    except:
        pass
    return None

df_tk['hashtag'] = df_tk['input'].apply(extract_hashtag)

# Aggregate metrics per hashtag
tk_agg = df_tk.groupby('hashtag').agg({
    'playCount': 'sum', 
    'diggCount': 'sum', 
    'shareCount': 'sum'
}).reset_index()

# Fuzzy Map
def fuzzy_map_tk(hashtag):
    if pd.isna(hashtag): return None
    clean_h = str(hashtag).lower().replace('restaurant', '').replace('bkk', '')
    match, score = process.extractOne(clean_h, choices)
    if score >= 60:
        return match
    return None

tk_agg['matched_name'] = tk_agg['hashtag'].apply(fuzzy_map_tk)

# Re-aggregate matched names (multiple hashtags might map to the same restaurant)
tk_final = tk_agg.groupby('matched_name').sum(numeric_only=True).reset_index()
tk_final.rename(columns={
    'playCount': 'tiktok_views', 
    'diggCount': 'tiktok_likes', 
    'shareCount': 'tiktok_shares'
}, inplace=True)
print(f"Extracted {len(tk_final)} unique mapped TikTok records.")


--- Processing TikTok ---


Extracted 0 unique mapped TikTok records.


## 4. Facebook & Wongnai (Via Google Search JSON Profile Parsing)

In [5]:
print("\n--- Processing Facebook & Wongnai ---")

df_fb = pd.read_csv("Apify/apify_facebook_test.csv")
df_wn = pd.read_csv("Apify/apify_wongnai_test.csv")

def parse_fb(row):
    try:
        res = ast.literal_eval(row)
        for r in res:
            if 'facebook.com' in r.get('url', ''):
                s = r.get('text', '')
                likes = re.search(r'([\d,K]+)\s+likes', s)
                if likes:
                    val = likes.group(1).replace(',', '')
                    if 'K' in val: 
                        return float(val.replace('K','')) * 1000
                    return float(val)
    except:
        pass
    return 0

def parse_wn(row):
    try:
        res = ast.literal_eval(row)
        for r in res:
            if 'wongnai.com' in r.get('url', ''):
                s = r.get('text', '')
                rat = re.search(r'Rating:\s+([\d.]+)', s)
                rev = re.search(r'([\d,]+)\s+reviews', s)
                r_val = float(rat.group(1)) if rat else 0
                rv_val = float(rev.group(1).replace(',', '')) if rev else 0
                return r_val, rv_val
    except:
        pass
    return 0, 0

# Facebook
df_fb['facebook_likes'] = df_fb['organicResults'].apply(parse_fb)
df_fb['matched_name'] = df_fb['searchQuery'].apply(lambda q: process.extractOne(str(q).replace(' Facebook', ''), choices)[0] if process.extractOne(str(q).replace(' Facebook', ''), choices)[1] >= 85 else None)
fb_final = df_fb.groupby('matched_name')['facebook_likes'].max().reset_index()

# Wongnai
df_wn[['wongnai_rating', 'wongnai_reviews']] = pd.DataFrame(df_wn['organicResults'].apply(parse_wn).tolist(), index=df_wn.index)
df_wn['matched_name'] = df_wn['searchQuery'].apply(lambda q: process.extractOne(str(q).replace(' wongnai', ''), choices)[0] if process.extractOne(str(q).replace(' wongnai', ''), choices)[1] >= 85 else None)
wn_final = df_wn.groupby('matched_name').agg({'wongnai_rating':'max', 'wongnai_reviews':'max'}).reset_index()

print(f"Extracted {len(fb_final)} Facebook records and {len(wn_final)} Wongnai records.")


--- Processing Facebook & Wongnai ---


Extracted 3 Facebook records and 375 Wongnai records.


## 5. Merge Outputs

In [6]:
final_scrape = df_master[['restaurant_id', 'restaurant_name_en']].copy()

# 1. Gmaps (Exact ID Merge)
final_scrape = pd.merge(final_scrape, df_gmaps, on='restaurant_id', how='left')

# 2. Socials (Fuzzy Name Merge)
final_scrape = pd.merge(final_scrape, tk_final, left_on='restaurant_name_en', right_on='matched_name', how='left').drop(columns=['matched_name'])
final_scrape = pd.merge(final_scrape, fb_final, left_on='restaurant_name_en', right_on='matched_name', how='left').drop(columns=['matched_name'])
final_scrape = pd.merge(final_scrape, wn_final, left_on='restaurant_name_en', right_on='matched_name', how='left').drop(columns=['matched_name'])

numeric_cols = final_scrape.select_dtypes(include=[np.number]).columns
final_scrape[numeric_cols] = final_scrape[numeric_cols].fillna(0)
out_file = os.path.join(OUTPUT_DIR, "master_web_scraping_data.csv")
final_scrape.to_csv(out_file, index=False)

print(f"\nâœ… Web Scraping Integration Complete! Saved {len(final_scrape)} rows to {out_file}")
final_scrape.head()


âœ… Web Scraping Integration Complete! Saved 3727 rows to Output_Data\master_web_scraping_data.csv


,restaurant_id,restaurant_name_en,gmaps_rating,gmaps_reviews,tiktok_views,tiktok_likes,tiktok_shares,facebook_likes,wongnai_rating,wongnai_reviews
0,33.0,Audrey Cafe Thonglor Soi 11,4.2,1161.0,0.0,0.0,0.0,0.0,0.0,0.0
1,34.0,Rang Mahal Rooftop at Rembrandt Hotel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,168.0,Attico Cucina Italiana Radisson Blu Plaza Bangkok,4.5,185.0,0.0,0.0,0.0,0.0,0.0,0.0
3,220.0,The Living Room at Sheraton Grande Sukhumvit A...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,222.0,Rain Tree Cafe at The Athenee Hotel,4.4,796.0,0.0,0.0,0.0,0.0,0.0,0.0
